# 03 — Analysis and plotting

The point of Python bindings: analyze a live simulation with **numpy** and
**matplotlib** while it runs. We melt an LJ crystal and watch thermodynamics,
mean-squared displacement (self-diffusion) and structure (RDF) respond.

In [ ]:
%pip install lammps-js matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps, LMP_STYLE_GLOBAL, LMP_STYLE_ATOM, LMP_TYPE_VECTOR, LMP_TYPE_ARRAY

lmp = await lammps(output=None)   # output=None silences the log
lmp.commands_string("""
units         lj
atom_style    atomic
lattice       fcc 0.8442
region        box block 0 4 0 4 0 4
create_box    1 box
create_atoms  1 box
mass          1 1.0
velocity      all create 3.0 87287
pair_style    lj/cut 2.5
pair_coeff    1 1 1.0 1.0 2.5
fix           1 all nve
compute       msd all msd
thermo        100
""")
print(lmp.get_natoms(), "atoms")

## Thermodynamics over time

Run in chunks and collect any thermo keyword with `get_thermo` — the Python
loop drives LAMMPS directly, no log-file parsing:

In [ ]:
steps, temps, pes, msds = [], [], [], []
for chunk in range(40):
    lmp.command("run 25")
    steps.append(lmp.extract_global("ntimestep"))
    temps.append(lmp.get_thermo("temp"))
    pes.append(lmp.get_thermo("pe"))
    # compute msd: current global vector is [dx2, dy2, dz2, dr2]
    msds.append(lmp.extract_compute("msd", LMP_STYLE_GLOBAL, LMP_TYPE_VECTOR)[3])

steps, temps, pes, msds = map(np.array, (steps, temps, pes, msds))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
ax1.plot(steps, temps)
ax1.set(xlabel="timestep", ylabel="temperature", title="Melting: T relaxes")
ax2.plot(steps, pes, color="tab:orange")
ax2.set(xlabel="timestep", ylabel="potential energy / atom")
fig.tight_layout()
plt.show()

## Self-diffusion from the MSD

In a liquid the mean-squared displacement grows linearly with time; the
Einstein relation gives the diffusion coefficient $D$ from the slope,
$\mathrm{MSD} = 6 D t$. Fit the tail (after melting):

In [ ]:
dt = lmp.extract_global("dt")
time = steps * dt
tail = slice(len(steps) // 2, None)
slope, intercept = np.polyfit(time[tail], msds[tail], 1)
D = slope / 6

plt.figure(figsize=(5, 3.2))
plt.plot(time, msds, label="MSD")
plt.plot(time[tail], slope * time[tail] + intercept, "--",
         label=f"fit: D = {D:.3f} (LJ units)")
plt.xlabel("time"); plt.ylabel("MSD"); plt.legend(); plt.tight_layout()
plt.show()

## Structure: radial distribution function

`compute rdf` + `extract_compute(..., LMP_TYPE_ARRAY)` returns the current
g(r) curve as a 2-column array (r, g):

In [ ]:
lmp.command("compute myrdf all rdf 100")
lmp.command("run 100")

rdf = lmp.extract_compute("myrdf", LMP_STYLE_GLOBAL, LMP_TYPE_ARRAY)
plt.figure(figsize=(5, 3.2))
plt.plot(rdf[:, 0], rdf[:, 1])
plt.xlabel("r"); plt.ylabel("g(r)"); plt.title("Liquid structure")
plt.tight_layout(); plt.show()

## Per-atom analysis

Per-atom computes come back as numpy arrays too — here the kinetic energy
distribution across atoms (Maxwell–Boltzmann):

In [ ]:
lmp.command("compute ke all ke/atom")
lmp.command("run 0 post no")

ke = lmp.extract_compute("ke", LMP_STYLE_ATOM, LMP_TYPE_VECTOR)
plt.figure(figsize=(5, 3.2))
plt.hist(ke, bins=40, density=True)
plt.xlabel("kinetic energy per atom"); plt.ylabel("probability density")
plt.tight_layout(); plt.show()

lmp.close()

Next: [04 — Multithreading with KOKKOS](04-multithreading-kokkos.ipynb).